In [ ]:
import pandas as pd

In [ ]:
# Load Datasets

rate_df = pd.read_csv("withGDP_Merged_macrovars.csv")
sentiment_df = pd.read_csv("econobert_speeches.csv")

In [ ]:
sentiment_df.info()
sentiment_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 676 entries, 0 to 675
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   index           676 non-null    int64  
 1   Date            676 non-null    object 
 2   tone_mean       676 non-null    float64
 3   tone_median     676 non-null    float64
 4   pos_share       676 non-null    float64
 5   neg_share       676 non-null    float64
 6   neu_share       676 non-null    float64
 7   n_sent          676 non-null    int64  
 8   label_majority  676 non-null    object 
dtypes: float64(5), int64(2), object(2)
memory usage: 47.7+ KB


,index,Date,tone_mean,tone_median,pos_share,neg_share,neu_share,n_sent,label_majority
0,0,2020-03-03,0.569225,0.929719,0.670103,0.082474,0.237113,97,positive
1,1,2020-02-28,0.571175,0.840616,0.624000,0.040000,0.336000,125,positive
2,2,2020-02-27,0.601772,0.934495,0.718750,0.078125,0.195312,128,positive
3,3,2020-02-06,0.491022,0.852925,0.613636,0.090909,0.272727,88,positive
4,4,2020-01-31,0.126351,0.059077,0.100000,0.050000,0.850000,20,neutral


In [ ]:
#dropping unnecessary columns for the succeeding analysis
sentiment_df = sentiment_df.drop(columns=[
    'tone_median',
    'n_sent',
    'label_majority'
    ])
sentiment_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 676 entries, 0 to 675
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   index      676 non-null    int64  
 1   Date       676 non-null    object 
 2   tone_mean  676 non-null    float64
 3   pos_share  676 non-null    float64
 4   neg_share  676 non-null    float64
 5   neu_share  676 non-null    float64
dtypes: float64(4), int64(1), object(1)
memory usage: 31.8+ KB


In [ ]:
rate_df.info()
rate_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 286 entries, 0 to 285
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Date                      286 non-null    object 
 1   INTERBANK CALL LOAN RATE  286 non-null    float64
 2   Real GDP                  286 non-null    float64
 3   CPI                       286 non-null    float64
 4   Wholesale Price           286 non-null    float64
 5   Industrial Production     286 non-null    float64
 6   Intl Trade Merch Exports  286 non-null    float64
 7   Intl Trade Merch Imports  286 non-null    float64
 8   FX Rate                   286 non-null    float64
dtypes: float64(8), object(1)
memory usage: 20.2+ KB


,Date,INTERBANK CALL LOAN RATE,Real GDP,CPI,Wholesale Price,Industrial Production,Intl Trade Merch Exports,Intl Trade Merch Imports,FX Rate
0,2001-04-01,9.875000,2.414138,7.551018,7.214612,21.064873,13.683626,31.135206,21.843502
1,2001-05-01,9.343750,3.268185,7.426241,8.432629,15.650250,2.635522,49.747061,20.888137
2,2001-06-01,9.092187,3.268185,7.411171,8.469950,10.694178,7.047290,45.582767,20.725388
3,2001-07-01,9.196023,3.268185,7.236184,9.882138,8.655032,-8.622024,38.405294,19.994097
4,2001-08-01,9.264205,3.110441,7.400007,9.784558,7.508969,-3.500603,28.565700,15.791568


In [ ]:
def to_month_start(s: pd.Series) -> pd.Series:
    """Convert date-like series to month-start timestamps (monthly frequency)."""
    return pd.to_datetime(s, errors="coerce").dt.to_period("M").dt.to_timestamp()

# 1) MONTHLY backbone = rate_df (keep ALL columns)
rate = rate_df.copy()
rate["Date"] = pd.to_datetime(rate["Date"], errors="coerce")
rate = rate.dropna(subset=["Date"])
rate["Date"] = to_month_start(rate["Date"])

In [ ]:
date_counts = rate['Date'].value_counts()
if (date_counts > 1).any():
    print("Multiple rows exist for some months. Here are the months with more than one entry:")
    print(date_counts[date_counts > 1])
else:
    print("Each month has exactly one row in the `rate` DataFrame.")

Each month has exactly one row in the `rate` DataFrame.


In [ ]:
# Ensure sorted (optional)
rate = rate.sort_values("Date").reset_index(drop=True)

In [ ]:
# 2) Aggregate sentiment_df to MONTHLY values
#    - multiple speeches in month -> average value

sent = sentiment_df.loc[:, ["Date", "tone_mean", "pos_share", "neg_share", "neu_share"]].copy()
sent["Date"] = pd.to_datetime(sent["Date"], errors="coerce")
sent = sent.dropna(subset=["Date"])

sent_monthly = (
    sent.assign(Date=to_month_start(sent["Date"]))
        .groupby("Date", as_index=False)
        .agg(
            tone_mean=("tone_mean", "mean"),
            pos_mean=("pos_share", "mean"),
            neg_mean=("neg_share", "mean"),
            neu_mean=("neu_share", "mean"),
            speech_count=("tone_mean", "size")
        )
)

In [ ]:
# 3) Merge: keep MONTHLY macro data even if no speeches (LEFT JOIN onto rate)

merged = rate.merge(sent_monthly, on="Date", how="left").sort_values("Date").reset_index(drop=True)

In [ ]:
merged

,Date,INTERBANK CALL LOAN RATE,Real GDP,CPI,Wholesale Price,Industrial Production,Intl Trade Merch Exports,Intl Trade Merch Imports,FX Rate,tone_mean,pos_mean,neg_mean,neu_mean,speech_count
0,2001-04-01,9.875000,2.414138,7.551018,7.214612,21.064873,13.683626,31.135206,21.843502,0.419630,0.507311,0.072353,0.415574,5.0
1,2001-05-01,9.343750,3.268185,7.426241,8.432629,15.650250,2.635522,49.747061,20.888137,0.537493,0.631274,0.067807,0.289641,5.0
2,2001-06-01,9.092187,3.268185,7.411171,8.469950,10.694178,7.047290,45.582767,20.725388,0.435455,0.459594,0.009009,0.531397,3.0
3,2001-07-01,9.196023,3.268185,7.236184,9.882138,8.655032,-8.622024,38.405294,19.994097,0.416602,0.488973,0.067964,0.439554,5.0
4,2001-08-01,9.264205,3.110441,7.400007,9.784558,7.508969,-3.500603,28.565700,15.791568,0.401708,0.532059,0.115389,0.345991,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
281,2024-10-01,6.244332,5.239308,1.937047,1.973682,-6.966527,-8.713099,8.727526,0.901607,NaN,NaN,NaN,NaN,NaN
282,2024-11-01,6.159405,5.252809,2.263543,2.118341,-1.599997,-3.973003,12.816407,5.164532,NaN,NaN,NaN,NaN,NaN
283,2024-12-01,6.086828,5.252809,2.502016,2.261117,-4.130874,-3.839992,1.652797,5.145626,NaN,NaN,NaN,NaN,NaN
284,2025-01-01,5.895856,5.252809,2.900886,2.696791,0.079971,2.981743,3.488013,4.320615,NaN,NaN,NaN,NaN,NaN


In [ ]:
merged[merged['tone_mean'].isna()]

,Date,INTERBANK CALL LOAN RATE,Real GDP,CPI,Wholesale Price,Industrial Production,Intl Trade Merch Exports,Intl Trade Merch Imports,FX Rate,tone_mean,pos_mean,neg_mean,neu_mean,speech_count
10,2002-02-01,7.498355,3.876541,3.714286,5.484146,-8.365021,-7.668831,-30.090635,6.195906,NaN,NaN,NaN,NaN,NaN
14,2002-06-01,7.125003,3.660185,3.497161,5.625527,4.566641,10.817769,-1.057510,-2.100636,NaN,NaN,NaN,NaN,NaN
21,2003-01-01,7.067434,4.565113,2.488482,6.456956,10.654207,14.024249,18.836364,4.189519,NaN,NaN,NaN,NaN,NaN
22,2003-02-01,7.060855,4.618477,2.754824,6.255075,14.834025,8.320999,56.766683,5.446697,NaN,NaN,NaN,NaN,NaN
23,2003-03-01,7.117564,4.618477,3.219869,6.467255,11.165525,11.927679,37.382946,6.902442,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
281,2024-10-01,6.244332,5.239308,1.937047,1.973682,-6.966527,-8.713099,8.727526,0.901607,NaN,NaN,NaN,NaN,NaN
282,2024-11-01,6.159405,5.252809,2.263543,2.118341,-1.599997,-3.973003,12.816407,5.164532,NaN,NaN,NaN,NaN,NaN
283,2024-12-01,6.086828,5.252809,2.502016,2.261117,-4.130874,-3.839992,1.652797,5.145626,NaN,NaN,NaN,NaN,NaN
284,2025-01-01,5.895856,5.252809,2.900886,2.696791,0.079971,2.981743,3.488013,4.320615,NaN,NaN,NaN,NaN,NaN


In [ ]:
# 4) Fill months with no speeches by copying the previous available mean

merged["tone_mean_filled"] = merged["tone_mean"].ffill()
merged["pos_mean_filled"] = merged["pos_mean"].ffill()
merged["neg_mean_filled"] = merged["neg_mean"].ffill()
merged["neu_mean_filled"] = merged["neu_mean"].ffill()

# Make speech_count explicit (0 if no speeches that month)
merged["speech_count"] = merged["speech_count"].fillna(0).astype(int)

In [ ]:
merged[merged['tone_mean'].isna()]

,Date,INTERBANK CALL LOAN RATE,Real GDP,CPI,Wholesale Price,Industrial Production,Intl Trade Merch Exports,Intl Trade Merch Imports,FX Rate,tone_mean,pos_mean,neg_mean,neu_mean,speech_count,tone_mean_filled,pos_mean_filled,neg_mean_filled,neu_mean_filled
10,2002-02-01,7.498355,3.876541,3.714286,5.484146,-8.365021,-7.668831,-30.090635,6.195906,NaN,NaN,NaN,NaN,0,0.271050,0.458333,0.145833,0.354167
14,2002-06-01,7.125003,3.660185,3.497161,5.625527,4.566641,10.817769,-1.057510,-2.100636,NaN,NaN,NaN,NaN,0,0.301911,0.387022,0.063307,0.549671
21,2003-01-01,7.067434,4.565113,2.488482,6.456956,10.654207,14.024249,18.836364,4.189519,NaN,NaN,NaN,NaN,0,0.516605,0.543103,0.017241,0.439655
22,2003-02-01,7.060855,4.618477,2.754824,6.255075,14.834025,8.320999,56.766683,5.446697,NaN,NaN,NaN,NaN,0,0.516605,0.543103,0.017241,0.439655
23,2003-03-01,7.117564,4.618477,3.219869,6.467255,11.165525,11.927679,37.382946,6.902442,NaN,NaN,NaN,NaN,0,0.516605,0.543103,0.017241,0.439655
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
281,2024-10-01,6.244332,5.239308,1.937047,1.973682,-6.966527,-8.713099,8.727526,0.901607,NaN,NaN,NaN,NaN,0,0.569225,0.670103,0.082474,0.237113
282,2024-11-01,6.159405,5.252809,2.263543,2.118341,-1.599997,-3.973003,12.816407,5.164532,NaN,NaN,NaN,NaN,0,0.569225,0.670103,0.082474,0.237113
283,2024-12-01,6.086828,5.252809,2.502016,2.261117,-4.130874,-3.839992,1.652797,5.145626,NaN,NaN,NaN,NaN,0,0.569225,0.670103,0.082474,0.237113
284,2025-01-01,5.895856,5.252809,2.900886,2.696791,0.079971,2.981743,3.488013,4.320615,NaN,NaN,NaN,NaN,0,0.569225,0.670103,0.082474,0.237113


In [ ]:
merged[merged['tone_mean_filled'].isna()]

,Date,INTERBANK CALL LOAN RATE,Real GDP,CPI,Wholesale Price,Industrial Production,Intl Trade Merch Exports,Intl Trade Merch Imports,FX Rate,tone_mean,pos_mean,neg_mean,neu_mean,speech_count,tone_mean_filled,pos_mean_filled,neg_mean_filled,neu_mean_filled


In [ ]:
print(f"Rate DataFrame Date Range: {rate['Date'].min().date()} to {rate['Date'].max().date()}")
print(f"Sentiment Monthly DataFrame Date Range: {sent_monthly['Date'].min().date()} to {sent_monthly['Date'].max().date()}")

Rate DataFrame Date Range: 2001-04-01 to 2025-02-01
Sentiment Monthly DataFrame Date Range: 1998-04-01 to 2020-03-01


In [ ]:
start_date = '2001-04-01'
end_date = '2020-03-01'

# Filter the DataFrame
merged = merged[(merged['Date'] >= start_date) & (merged['Date'] <= end_date)]

print(f"Filtered Merged DataFrame Shape: {merged.shape}")
print(f"Filtered Merged DataFrame Date Range: {merged['Date'].min().date()} to {merged['Date'].max().date()}")

Filtered Merged DataFrame Shape: (228, 18)
Filtered Merged DataFrame Date Range: 2001-04-01 to 2020-03-01


In [ ]:
merged.head()

,Date,INTERBANK CALL LOAN RATE,Real GDP,CPI,Wholesale Price,Industrial Production,Intl Trade Merch Exports,Intl Trade Merch Imports,FX Rate,tone_mean,pos_mean,neg_mean,neu_mean,speech_count,tone_mean_filled,pos_mean_filled,neg_mean_filled,neu_mean_filled
0,2001-04-01,9.875000,2.414138,7.551018,7.214612,21.064873,13.683626,31.135206,21.843502,0.419630,0.507311,0.072353,0.415574,5,0.419630,0.507311,0.072353,0.415574
1,2001-05-01,9.343750,3.268185,7.426241,8.432629,15.650250,2.635522,49.747061,20.888137,0.537493,0.631274,0.067807,0.289641,5,0.537493,0.631274,0.067807,0.289641
2,2001-06-01,9.092187,3.268185,7.411171,8.469950,10.694178,7.047290,45.582767,20.725388,0.435455,0.459594,0.009009,0.531397,3,0.435455,0.459594,0.009009,0.531397
3,2001-07-01,9.196023,3.268185,7.236184,9.882138,8.655032,-8.622024,38.405294,19.994097,0.416602,0.488973,0.067964,0.439554,5,0.416602,0.488973,0.067964,0.439554
4,2001-08-01,9.264205,3.110441,7.400007,9.784558,7.508969,-3.500603,28.565700,15.791568,0.401708,0.532059,0.115389,0.345991,5,0.401708,0.532059,0.115389,0.345991


In [ ]:
merged.to_csv('withGDP_IRD_with_tone_dataset.csv', index=False)